## Initial networks for baseline comparison

In [ ]:
from util import *
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from time import perf_counter

# Data generation
np.random.seed(2025)
n_points = 50

noise = 0.01 * np.random.randn(n_points,1)
x = np.linspace(-1, 1, n_points).reshape(-1,1)
y = Runge(x) + noise

degree = 14
X = PolynomialFeatures(degree).fit_transform(x)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

# Data scaling
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
y_offset = np.mean(y_test)

# Do a bit of a hack to work with the exact same data as the regression does :)
x_train_s = X_train_s[:,1]
x_test_s = X_test_s[:,1]
x_train_s = x_train_s.reshape(-1,1)
y_train = y_train.reshape(-1,1)
# OLS regression
t=perf_counter()
beta_ols = OLS(X_train_s, y_train)
y_pred_ols = X_test_s @ beta_ols + y_offset
print("OLS training time:", perf_counter()-t)
mse_ols = mean_squared_error(y_test, y_pred_ols)
r2_ols = r2_score(y_test, y_pred_ols)
print("OLS MSE",mse_ols)
print("OLS R2",r2_ols)
print()

# 2 layer NN
print("Training 2 layer NN")
t=perf_counter()
nn2 = NeuralNetwork(1, [100, 100, 1], [sigmoid, sigmoid, linear], [sigmoid_der, sigmoid_der, linear_der], MSE, MSE_der)
nn2.RMSProp_stochastic(x_train_s, y_train-y_offset, learning_rate = 0.01, rho = 0.99, epochs = 5000, minibatch_size=5)
nn2_pred = nn2.predict_batch(x_test_s.reshape(-1,1)) + y_offset
print("NN2 training time:", perf_counter()-t)
mse_nn2 = mean_squared_error(y_test, nn2_pred)
r2_nn2 = r2_score(y_test, nn2_pred)
print("NN2 MSE",mse_nn2)
print("NN2 R2",r2_nn2)
print()

# 1 layer NN
print("Training 1 layer NN")
t=perf_counter()
nn1 = NeuralNetwork(1, [50, 1], [sigmoid, linear], [sigmoid_der, linear_der], MSE, MSE_der)
nn1.RMSProp_stochastic(x_train_s, y_train-y_offset, learning_rate = 0.01, rho = 0.99, epochs = 5000, minibatch_size = 5)
nn1_pred = nn1.predict_batch(x_test_s.reshape(-1,1)) + y_offset
print("NN1 training time:", perf_counter()-t)
mse_nn1 = mean_squared_error(y_test, nn1_pred)
r2_nn1 = r2_score(y_test, nn1_pred)
print("NN1 OLS", mse_nn1)
print("NN1 R2",r2_nn1)


## Compare simple 1 and 2 layer NN with Pytorch

In [ ]:
# --- PyTorch equivalents for the 1-layer and 2-layer NNs ---
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from util import *
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

def OLS(X:np.ndarray, y:np.ndarray)->np.ndarray:
    return np.linalg.pinv(X.T @ X) @ X.T @ y

# Data generation
np.random.seed(2025)
n_points = 50

noise = 0.01 * np.random.randn(n_points,1)
x = np.linspace(-1, 1, n_points).reshape(-1,1)
y = Runge(x) + noise

degree = 14
X = PolynomialFeatures(degree).fit_transform(x)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

# Data scaling
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
y_offset = np.mean(y_test)

# Do a bit of a hack to work with the exact same data as the regression does :)
x_train_s = X_train_s[:,1]
x_test_s = X_test_s[:,1]

torch.manual_seed(2025)

# Prepare tensors
x_train_t = torch.tensor(x_train_s.reshape(-1, 1), dtype=torch.float32)
y_train_t = torch.tensor((y_train - y_offset).reshape(-1, 1), dtype=torch.float32)
x_test_t  = torch.tensor(x_test_s.reshape(-1, 1), dtype=torch.float32)

# Function to train PyTorch models
def train_torch_model(model, x_t, y_t, epochs=5000, batch_size=5, lr=0.01, rho=0.99):
    dataset = TensorDataset(x_t, y_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    criterion = nn.MSELoss()
    optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, alpha=rho, centered=False)

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

# 1-layer NN: [1] -> [50, sigmoid] -> [1, linear]
print("Training 1 layer NN (PyTorch)")
nn1_torch = nn.Sequential(nn.Linear(1, 50), nn.Sigmoid(), nn.Linear(50, 1))
train_torch_model(nn1_torch, x_train_t, y_train_t, epochs=5000, batch_size=5, lr=0.01, rho=0.99)

nn1_torch.eval()
with torch.no_grad():
    nn1_pred_t = nn1_torch(x_test_t).cpu().numpy() + y_offset
mse_nn1_t = mean_squared_error(y_test, nn1_pred_t)
r2_nn1_t = r2_score(y_test, nn1_pred_t)
print("NN1 MSE (PyTorch)", mse_nn1_t)
print("NN1 R2 (PyTorch)", r2_nn1_t)
print()
# 2-layer NN: [1] -> [100, sigmoid] -> [100, sigmoid] -> [1, linear]
print("Training 2 layer NN (PyTorch)")
nn2_torch = nn.Sequential(
    nn.Linear(1, 100),
    nn.Sigmoid(),
    nn.Linear(100, 100),
    nn.Sigmoid(),
    nn.Linear(100, 1)
)
train_torch_model(nn2_torch, x_train_t, y_train_t, epochs=5000, batch_size=5, lr=0.01, rho=0.99)

nn2_torch.eval()
with torch.no_grad():
    nn2_pred_t = nn2_torch(x_test_t).cpu().numpy() + y_offset
mse_nn2_t = mean_squared_error(y_test, nn2_pred_t)
r2_nn2_t = r2_score(y_test, nn2_pred_t)
print("NN2 MSE (PyTorch)", mse_nn2_t)
print("NN2 R2 (PyTorch)", r2_nn2_t)

## Learning rate analysis

In [ ]:
from util import *
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

np.random.seed(2025)
n_points = 1000

noise = 0.01 * np.random.randn(n_points,1)
x = np.linspace(-1, 1, n_points).reshape(-1,1)
y = Runge(x) + noise

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25)
scaler = StandardScaler()
scaler.fit(x_train)
x_train_s = scaler.transform(x_train)
x_test_s = scaler.transform(x_test)
y_offset = np.mean(y_test)

learning_rates = [0.0001, 0.001, 0.01, 0.05]
network_input_size = 1
layer_output_sizes = [100, 100, 1]
activation_funcs = [sigmoid, sigmoid, linear]
activation_ders = [sigmoid_der, sigmoid_der, linear_der]
cost_fun = MSE
cost_der = MSE_der

nn = NeuralNetwork(network_input_size, layer_output_sizes, activation_funcs, activation_ders, cost_fun, cost_der)

mses = np.zeros((4,len(learning_rates)))
r2s = np.zeros((4,len(learning_rates)))
for i, lr in enumerate(learning_rates):
    # Plain GD
    print("Training plain GD with lr", lr)
    nn.reset()
    nn.gradient_descent(x_train_s, y_train - y_offset, learning_rate=lr, epochs=500)
    nn_pred = nn.predict_batch(x_test_s) + y_offset 
    mse = mean_squared_error(y_test, nn_pred)
    r2 = r2_score(y_test, nn_pred)
    mses[0, i] = mse
    r2s[0, i] = r2
    print("MSE:", mse, "R2:", r2, "\n")

    # RMSProp
    print("Training RMSProp with lr", lr)
    nn.reset()
    nn.RMSProp(x_train_s, y_train - y_offset, learning_rate=lr, rho=0.99, epochs=500)
    nn_pred = nn.predict_batch(x_test_s) + y_offset
    mse = mean_squared_error(y_test, nn_pred)
    r2 = r2_score(y_test, nn_pred)
    mses[1, i] = mse
    r2s[1, i] = r2
    print("MSE:", mse, "R2:", r2, "\n")
    
    # Stochastic RMSProp
    print("Training RMSProp SGD with lr", lr)
    nn.reset()
    nn.RMSProp_stochastic(x_train_s, y_train - y_offset, learning_rate=lr, rho=0.99, epochs=500, minibatch_size=5)
    nn_pred = nn.predict_batch(x_test_s) + y_offset
    mse = mean_squared_error(y_test, nn_pred)
    r2 = r2_score(y_test, nn_pred)
    mses[2, i] = mse
    r2s[2, i] = r2
    print("MSE:", mse, "R2:", r2, "\n")


    # Stochastic ADAM
    print("Training ADAM SGD with lr", lr)
    nn.reset()
    nn.ADAM_stochastic(x_train_s, y_train - y_offset, learning_rate=lr, rho1=0.9, rho2=0.999, epochs=500, minibatch_size=5)
    nn_pred = nn.predict_batch(x_test_s) + y_offset
    mse = mean_squared_error(y_test, nn_pred)
    r2 = r2_score(y_test, nn_pred)
    mses[3, i] = mse
    r2s[3, i] = r2
    print("MSE:", mse, "R2:", r2, "\n")

### Plotting

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure()
sns.heatmap(mses, cmap = "rocket", annot=True, fmt = ".4f", xticklabels=learning_rates, yticklabels=["Gradient Descent", "RMSProp stochastic", "ADAM stochastic"])
plt.xlabel("Learning Rate")
plt.ylabel("Optimizer")
plt.title("MSE optimizers and learning rates")
plt.show()

plt.figure()
sns.heatmap(r2s, cmap = "rocket", annot=True, fmt = ".3f", xticklabels=learning_rates, yticklabels=["Gradient Descent", "RMSProp stochastic", "ADAM stochastic"])
plt.xlabel("Learning Rate")
plt.ylabel("Optimizer")
plt.title(r"$R^2$ Score")
plt.show()